<a href="https://colab.research.google.com/github/DivyaMeenaSundaram/Prompt-Engineering/blob/main/Prompt_versioning_json.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# os → creates folders and files
# json → saves metadata as JSON
# hashlib → creates a unique fingerprint of the prompt
# datetime → records when the version was created
import os
import json
import hashlib
from datetime import datetime

In [2]:
# Create folders for storing prompt text and metadata

base_folder = "prompt_versioning_demo"

prompt_folder = os.path.join(base_folder, "prompts")
metadata_folder = os.path.join(base_folder, "metadata")

os.makedirs(prompt_folder, exist_ok=True)
os.makedirs(metadata_folder, exist_ok=True)

print("Folders created successfully.")

Folders created successfully.


In [3]:
# The same input will be used for all prompt versions.
# This makes comparison between versions more meaningful.

customer_question = """
I ordered a pair of headphones last week.
They arrived yesterday, but the left side isn't working.
What can I do?
"""

In [19]:
# Define the model configuration in ONE place.
# The same configuration will be used for the API call
# and saved in the metadata JSON file.

model_config = {
    "model": "gemini-3.6-flash",
    "temperature": 0.3,
    "max_output_tokens": 500,
    "top_p": 0.95,
    "top_k": 40
}

In [20]:
# Prompt Version 1
# Initial customer-service instruction

prompt_v1 = """
You are a customer service assistant.

Customer query:
{question}

Respond professionally and helpfully.
"""

In [21]:
# Prompt Version 2
# Change introduced:
# Responses should now be concise and empathetic.

prompt_v2 = """
You are a customer service assistant.

Customer query:
{question}

Respond professionally, concisely, and empathetically.
"""

In [22]:
# Prompt Version 3
# Additional business constraint and response structure.

prompt_v3 = """
You are a customer service assistant.

Customer query:
{question}

Respond professionally, concisely, and empathetically.

Do not promise a refund or replacement unless
the customer's eligibility has been established.

Structure the response as:
1. Acknowledge the problem
2. Explain the next step
3. Mention any information the customer needs to provide
"""

In [23]:
# Store all prompt versions together.
# This makes it easy to process them using the same code.

prompt_versions = {
    "v1": prompt_v1,
    "v2": prompt_v2,
    "v3": prompt_v3
}

In [24]:
prompt_versions["v2"]

'\nYou are a customer service assistant.\n\nCustomer query:\n{question}\n\nRespond professionally, concisely, and empathetically.\n'

In [25]:
# Create a unique fingerprint for the prompt text.
# Even a small change in the prompt will produce
# a different hash.

def create_prompt_hash(prompt):
    return hashlib.sha256(
        prompt.encode("utf-8")
    ).hexdigest()[:12]

In [26]:
for version, prompt in prompt_versions.items():

    prompt_hash = create_prompt_hash(prompt)

    print(version, "→", prompt_hash)

v1 → 935ca060938f
v2 → e2c5b23c1456
v3 → e947227961ec


In [27]:
# Save each prompt version as a separate text file.

for version, prompt in prompt_versions.items():

    file_path = os.path.join(
        prompt_folder,
        f"prompt_{version}.txt"
    )

    with open(file_path, "w", encoding="utf-8") as file:
        file.write(prompt.strip())

print("All prompt versions have been saved.")

All prompt versions have been saved.


In [28]:
# Store metadata associated with each prompt version.

change_reasons = {
    "v1": "Initial customer-service prompt",
    "v2": "Added conciseness and empathy",
    "v3": "Added eligibility constraint and response structure"
}

for version, prompt in prompt_versions.items():

    metadata = {
        "prompt_version": version,
        "prompt_hash": create_prompt_hash(prompt),

        "created_at": datetime.now().isoformat(),

        "change_reason": change_reasons[version],

        "model_configuration": model_config,

        "template_variables": ["question"],

        "status": "experimental"
    }

    json_path = os.path.join(
        metadata_folder,
        f"prompt_{version}.json"
    )

    with open(json_path, "w", encoding="utf-8") as file:
        json.dump(metadata, file, indent=4)

print("Metadata JSON files saved.")

Metadata JSON files saved.


In [29]:
# Import Gemini only when you are ready to run the prompts.

import google.generativeai as genai

In [ ]:
# Enter the API key only when you want to generate responses.

API_KEY = input("Enter your Gemini API key: ")

genai.configure(api_key=API_KEY)

model = genai.GenerativeModel(
    model_config["model"]
)

In [31]:
# Insert the same customer question into each template.

final_prompts = {}

for version, prompt in prompt_versions.items():

    final_prompts[version] = prompt.format(
        question=customer_question
    )

print("Final prompts prepared.")

Final prompts prepared.


In [32]:
# Choose which prompt version to test.
# Change this to "v2" or "v3" when needed.

selected_version = "v3"

response = model.generate_content(
    final_prompts[selected_version],
    generation_config={
        "temperature": model_config["temperature"],
        "max_output_tokens": model_config["max_output_tokens"],
        "top_p": model_config["top_p"],
        "top_k": model_config["top_k"]
    }
)

print("PROMPT VERSION:", selected_version)
print("\nOUTPUT:")
print(response.text)

PROMPT VERSION: v3

OUTPUT:
Hello, 

I am so sorry to hear that your new headphones arrived with a faulty


In [33]:
import time

# Record the time taken to generate the response.
start_time = time.time()

response = model.generate_content(
    final_prompts[selected_version],
    generation_config={
        "temperature": model_config["temperature"],
        "max_output_tokens": model_config["max_output_tokens"],
        "top_p": model_config["top_p"],
        "top_k": model_config["top_k"]
    }
)

latency = time.time() - start_time

print(response.text)
print("\nLatency:", round(latency, 2), "seconds")

 formatting to align with the prompt's structure without looking unnatural):**

    Dear Customer,

Latency: 4.36 seconds


In [34]:
# Create a record of this particular model execution.

run_data = {
    "run_id": "run_001",

    "prompt_version": selected_version,

    "prompt_hash": create_prompt_hash(
        prompt_versions[selected_version]
    ),

    "model_configuration": model_config,

    "input": customer_question,

    "output": response.text,

    "latency_seconds": round(latency, 2),

    "timestamp": datetime.now().isoformat()
}

run_path = os.path.join(
    base_folder,
    f"{run_data['run_id']}_{selected_version}.json"
)

with open(run_path, "w", encoding="utf-8") as file:
    json.dump(run_data, file, indent=4)

print("Run information saved to:")
print(run_path)

Run information saved to:
prompt_versioning_demo/run_001_v3.json
